In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
df = pd.read_csv('fashion-mnist_train.csv')
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
46260,2,0,0,0,0,0,0,0,0,0,...,17,25,0,0,181,147,32,0,0,0
38598,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
46489,0,0,0,0,0,1,0,0,0,26,...,0,0,0,0,0,0,0,0,0,0
46159,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52524,4,0,0,0,0,0,0,0,0,0,...,0,0,0,136,179,66,0,0,0,0


In [5]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
from numpy import dtype
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):

        return len(self.features)

    def __getitem__(self, index):

        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)

In [10]:
test_dataset = CustomDataset(X_test, y_test)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [12]:
class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        return self.model(x)

In [13]:
epochs = 100
learning_rate = 0.1

In [14]:
model = MyNN(X_train.shape[1])
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [15]:
for epoch in range(epochs):

    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_epoch_loss += loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)

    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")




Epoch: 1, Loss: 0.6352872450451056
Epoch: 2, Loss: 0.4304826718171438
Epoch: 3, Loss: 0.3863647295186917
Epoch: 4, Loss: 0.35884156001607576
Epoch: 5, Loss: 0.3371040893346071
Epoch: 6, Loss: 0.3237453643058737
Epoch: 7, Loss: 0.3087256171206633
Epoch: 8, Loss: 0.29658989624430737
Epoch: 9, Loss: 0.2860280364875992
Epoch: 10, Loss: 0.27442648011197646
Epoch: 11, Loss: 0.2683351921240489
Epoch: 12, Loss: 0.25943088030442596
Epoch: 13, Loss: 0.25117718292027713
Epoch: 14, Loss: 0.244619861052682
Epoch: 15, Loss: 0.23960494832197826
Epoch: 16, Loss: 0.2320423257909715
Epoch: 17, Loss: 0.2259118615736564
Epoch: 18, Loss: 0.22264623193939526
Epoch: 19, Loss: 0.2167579050231725
Epoch: 20, Loss: 0.21066300708924732
Epoch: 21, Loss: 0.2071765677916507
Epoch: 22, Loss: 0.2004926116367181
Epoch: 23, Loss: 0.19556321803107857
Epoch: 24, Loss: 0.19249381325952708
Epoch: 25, Loss: 0.18825582519359887
Epoch: 26, Loss: 0.1836211814445754
Epoch: 27, Loss: 0.18163865377008914
Epoch: 28, Loss: 0.1739699

In [16]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [17]:
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    
print(correct/total)

0.886
